In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from imblearn.over_sampling import SMOTE, ADASYN, KMeansSMOTE
from imblearn.under_sampling import RandomUnderSampler, TomekLinks, NearMiss, ClusterCentroids
from imblearn.combine import SMOTEENN

# Load the dataset
df = pd.read_csv(r"T:\00 445\student\student-mat.csv", sep=";")

# Create binary target variable: 1 = Pass (G3 ≥ 10), 0 = Fail
df['pass'] = df['G3'].apply(lambda x: 1 if x >= 10 else 0)


#df.drop(columns=['G2','Medu','Fedu','studytime','famrel', 'G3'], inplace=True)
#df.drop(columns=['G2','G1', 'G3'], inplace=True)
#df.drop(columns=['G1','G2','Medu','Fedu','studytime','famrel', 'G3'], inplace=True)
df.drop(columns=['G2','G3'], inplace=True)
#df.drop(columns=['G3'], inplace=True)

# Separate features and target
X = df.drop(columns=['pass'])
y = df['pass']

# One-Hot Encoding for categorical columns
categorical_cols = X.select_dtypes(include='object').columns.tolist()
column_transformer = ColumnTransformer([("onehot", OneHotEncoder(drop='first'), categorical_cols)], remainder='passthrough')

# Transform the dataset
X_encoded = column_transformer.fit_transform(X)

# Convert the transformed data back to DataFrame
X_df = pd.DataFrame(X_encoded.toarray() if hasattr(X_encoded, 'toarray') else X_encoded)

# Split the dataset: 80% train, 5% validation, 15% test
X_train, X_temp, y_train, y_temp = train_test_split(X_df, y, test_size=0.30, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.55, stratify=y_temp, random_state=42)


# Flag to choose the sampling technique
SAMPLING_TECHNIQUE = 'ADASYN'
  # Options: 'SMOTE', 'ADASYN', 'SMOTEENN', 'UnderSampling', 'TomekLinks', 'NearMiss', 'ClusterCentroids', 'KMeansSMOTE'

# Apply selected sampling technique
if SAMPLING_TECHNIQUE == 'SMOTE':
    smote = SMOTE(random_state=42)
    X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)
    print("After SMOTE:")
    print(pd.Series(y_train_bal).value_counts())
elif SAMPLING_TECHNIQUE == 'ADASYN':
    adasyn = ADASYN(random_state=42)
    X_train_bal, y_train_bal = adasyn.fit_resample(X_train, y_train)
    print("After ADASYN:")
    print(pd.Series(y_train_bal).value_counts())
elif SAMPLING_TECHNIQUE == 'SMOTEENN':
    smote_enn = SMOTEENN(random_state=42)
    X_train_bal, y_train_bal = smote_enn.fit_resample(X_train, y_train)
    print("After SMOTEENN:")
    print(pd.Series(y_train_bal).value_counts())
elif SAMPLING_TECHNIQUE == 'UnderSampling':
    undersampler = RandomUnderSampler(random_state=42)
    X_train_bal, y_train_bal = undersampler.fit_resample(X_train, y_train)
    print("After Under-sampling:")
    print(pd.Series(y_train_bal).value_counts())
elif SAMPLING_TECHNIQUE == 'TomekLinks':
    tomek = TomekLinks()
    X_train_bal, y_train_bal = tomek.fit_resample(X_train, y_train)
    print("After Tomek Links:")
    print(pd.Series(y_train_bal).value_counts())
elif SAMPLING_TECHNIQUE == 'NearMiss':
    near_miss = NearMiss(version=1)  # Use version 1, 2, or 3
    X_train_bal, y_train_bal = near_miss.fit_resample(X_train, y_train)
    print("After NearMiss:")
    print(pd.Series(y_train_bal).value_counts())
elif SAMPLING_TECHNIQUE == 'ClusterCentroids':
    cluster_centroids = ClusterCentroids(random_state=42)
    X_train_bal, y_train_bal = cluster_centroids.fit_resample(X_train, y_train)
    print("After Cluster Centroids:")
    print(pd.Series(y_train_bal).value_counts())
elif SAMPLING_TECHNIQUE == 'KMeansSMOTE':
    kmeans_smote = KMeansSMOTE(random_state=42)
    X_train_bal, y_train_bal = kmeans_smote.fit_resample(X_train, y_train)
    print("After KMeansSMOTE:")
    print(pd.Series(y_train_bal).value_counts())
else:
    # If no sampling technique is selected, use the original training data
    X_train_bal, y_train_bal = X_train, y_train

# Standardize features for training, validation, and test sets
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_bal)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# You can now proceed with model training (e.g., Decision Tree, Random Forest, etc.)


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
# Step 1: Select only numerical columns
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns

# Calculate the correlation matrix for all numerical columns
correlation_matrix_all = df[numerical_cols].corr()

# Step 2: Plot the correlation matrix as a heatmap
plt.figure(figsize=(12, 10))  # Increase figure size for better readability
sns.heatmap(correlation_matrix_all, annot=True, cmap='coolwarm', fmt='.2f', cbar=True, annot_kws={'size': 12, 'weight': 'bold'}, 
            linewidths=0.5, linecolor='gray', vmin=-1, vmax=1)  # Added grid lines and color scale range
plt.title("Correlation Matrix of All Features", fontsize=16, weight='bold')
plt.xticks(rotation=45, ha='right', fontsize=12)  # Rotate x-axis labels for better visibility
plt.yticks(rotation=0, ha='right', fontsize=12)  # Keep y-axis labels horizontal for clarity
plt.tight_layout()  # Adjust layout to prevent overlap
plt.show()

# Step 3: Print the full correlation matrix
print("Correlation matrix of all features:")
print(correlation_matrix_all)
